# Folium

## Imports

In [ ]:
import json
import pathlib

import folium
import geopandas

## Carte et fonds de carte

Nantes se trouve à 47,218 de latitude et -1,554 de longitude. Affichez une carte Folium centrée sur Nantes, avec un niveau de zoom qui montre la ville, sur le fond topographique `OpenTopoMap` plutôt que sur le fond OpenStreetMap par défaut.

Ajoutez-y un second fond, la photographie aérienne `Esri.WorldImagery`, et un sélecteur de couches pour passer de l'un à l'autre. Que montre chaque fond que l'autre ne montre pas ?

**Documentation utile**

- https://python-visualization.github.io/folium/latest/reference.html#folium.raster_layers.TileLayer
- https://leaflet-extras.github.io/leaflet-providers/preview/

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `folium.Map(location=..., zoom_start=..., tiles=...)` : `location` est un couple (latitude, longitude), dans cet ordre ; `zoom_start` va de 0 (le monde entier) à 18 (la rue), et entre 12 et 14, la ville tient dans la carte ; `tiles` prend le nom d'un fond du paquet `xyzservices`, comme `"OpenTopoMap"`.
- `folium.TileLayer("Esri.WorldImagery", name=...)` crée un fond de plus, que `.add_to(carte)` ajoute à la carte ; `name` est son libellé dans le sélecteur.
- `folium.LayerControl()`, ajouté en dernier, liste les fonds de la carte : un seul s'affiche à la fois.
- Laissez de côté les fonds CartoDB : ils demandent désormais une clef d'API.
- La carte s'affiche quand elle est la dernière expression de la cellule.

```python
COORDONNEES_NANTES = 47.218371, -1.553621

carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=..., tiles=...)
folium.TileLayer(..., name="Photographie aérienne").add_to(carte)
folium.LayerControl().add_to(...)

carte
```

### Solution

In [ ]:
COORDONNEES_NANTES = 47.218371, -1.553621

carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=13, tiles="OpenTopoMap")
folium.TileLayer("Esri.WorldImagery", name="Photographie aérienne").add_to(carte)
folium.LayerControl().add_to(carte)

# Le fond topographique montre les noms, les routes et le relief ; la photographie,
# les bâtiments, la végétation et les bancs de sable de la Loire, sans aucun nom.
carte

## Ajout de marqueurs

Maintenant que nous disposons d'une carte correctement initialisée, nous allons y afficher des informations d'utilisation du service de vélos en libre-service nantais : Bicloo.

Commençons par récupérer les données avec la cellule ci-dessous.

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/nantes-open-data/infos-velos.json

### Chargement des données

Ces données sont maintenant disponibles dans le fichier `infos-velos.json`. Chargez ce fichier à l'aide du module `json` de la bibliothèque standard.

Le fichier compte 74 000 caractères, trop pour être lu à l'écran : affichez plutôt les clefs du dictionnaire chargé, le nombre de stations (la clef `nhits`), puis le premier élément de la liste `records`, une station. Où se trouve la position de la station ? Elle y figure deux fois : dans quel ordre chaque fois, et laquelle convient à `folium.Map(location=...)` ?

**Documentation utile**

- https://docs.python.org/3/library/json.html#json.load

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `json.load(fichier)` lit un fichier ouvert et renvoie l'objet Python qu'il contient : ici, un dictionnaire, dont `.keys()` donne les clefs.
- Ouvrez le fichier dans un `with`, avec `encoding="utf8"`.
- Le dictionnaire a trois clefs, `nhits`, `parameters` et `records` : `records` est la liste des 129 stations.
- Une station est un dictionnaire : `fields` contient ses champs (nom, places, vélos disponibles, position…), `geometry` sa géométrie au format GeoJSON.
- Comparez `fields["position"]` et `geometry["coordinates"]` aux coordonnées de Nantes : la latitude est vers 47, la longitude vers -1,5.

```python
with pathlib.Path("infos-velos.json").open(encoding="utf8") as fichier:
  donnees = ...
print(donnees.keys(), donnees[...])
premiere = donnees["records"][0]
print(premiere[...])
print(premiere["fields"][...], premiere["geometry"][...])
```

#### Solution

In [ ]:
with pathlib.Path("infos-velos.json").open(encoding="utf8") as fichier:
  donnees = json.load(fichier)
print(donnees.keys(), donnees["nhits"])
premiere = donnees["records"][0]
print(premiere["fields"])
# fields["position"] : [latitude, longitude], l'ordre qu'attend folium ;
# geometry["coordinates"] : [longitude, latitude], l'ordre de GeoJSON.
print(premiere["fields"]["position"], premiere["geometry"]["coordinates"])

### Création des marqueurs

Vous pouvez maintenant itérer sur les données (en particulier sur le contenu associé à la clef `records` de chaque élément des données) pour afficher un marqueur à chaque position de station Bicloo. Le nom de la station s'affiche au survol du marqueur par la souris.

**Documentation utile**

- https://python-visualization.github.io/folium/latest/reference.html#folium.map.Marker

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Chaque élément de `donnees["records"]` est une station : sa position est dans `station["fields"]["position"]`, une liste [latitude, longitude], l'ordre qu'attend folium (`station["geometry"]["coordinates"]` donne l'ordre inverse, longitude puis latitude, celui de GeoJSON).
- `folium.Marker(location=..., tooltip=...)` crée un marqueur, et sa méthode `.add_to(carte)` le pose sur la carte ; `tooltip` est le texte affiché au survol, ici `station["fields"]["name"]`.
- Créez une nouvelle carte avant la boucle, et terminez la cellule par `carte` pour l'afficher : 129 marqueurs.

```python
carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=14)

for station in donnees["records"]:
  folium.Marker(location=..., tooltip=...).add_to(carte)

carte
```

#### Solution

In [ ]:
carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=14)

for station in donnees["records"]:
  folium.Marker(
    location=station["fields"]["position"], tooltip=station["fields"]["name"]
  ).add_to(carte)

carte

### Personnalisation des marqueurs

Ajoutez un peu d'information à ces marqueurs :

- La couleur doit dépendre du nombre de vélos :
    - 0 → rouge
    - < 4 → orange
    - sinon vert
- Un clic sur un marqueur affiche l'adresse de la station et son nombre de places.

**Documentation utile**

- https://python-visualization.github.io/folium/latest/reference.html#folium.map.Icon

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Le nombre de vélos disponibles est dans `station["fields"]["available_bikes"]`, son adresse dans `station["fields"]["address"]`, son nombre de places dans `station["fields"]["bike_stands"]`.
- `popup=` accepte du HTML : `<br />` passe à la ligne.
- `folium.Marker` prend `popup=`, le texte affiché au clic, et `icon=folium.Icon(color=...)`, avec les couleurs `"red"`, `"orange"` et `"green"`.
- Vous devez obtenir 83 marqueurs verts, 32 orange et 14 rouges.

```python
carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=14)

for station in donnees["records"]:
  match station["fields"]["available_bikes"]:
    case x if x > 3:
      couleur = "green"
    case x if x > 0:
      couleur = ...
    case _:
      couleur = ...
  folium.Marker(
    location=station["fields"]["position"],
    popup=f"{...}<br />{...} places",
    icon=folium.Icon(color=couleur),
  ).add_to(carte)

carte
```

#### Solution

In [ ]:
carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=14)

for station in donnees["records"]:
  match station["fields"]["available_bikes"]:
    case x if x > 3:
      couleur = "green"
    case x if x > 0:
      couleur = "orange"
    case _:
      couleur = "red"
  folium.Marker(
    location=station["fields"]["position"],
    popup=f"{station['fields']['address']}<br />"
    f"{station['fields']['bike_stands']} places",
    icon=folium.Icon(color=couleur),
  ).add_to(carte)

carte

### Des cercles proportionnels (facultatif)

Remplacez les épingles par des cercles (`folium.CircleMarker`) dont le rayon suit le nombre de places de la station (`bike_stands`), bleus si la station a des vélos, rouges sinon, avec au survol le nom de la station, ses vélos disponibles et ses places.

Combien de stations n'ont aucun vélo ? Quelle est la plus grande station ? Une station n'apparaît pas : pourquoi ?

**Documentation utile**

- https://python-visualization.github.io/folium/latest/reference.html#folium.vector_layers.CircleMarker

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `folium.CircleMarker(location=..., radius=..., color=..., fill=True, tooltip=...)` : `radius` est en pixels, et le cercle garde sa taille à l'écran quand on zoome ; `fill=True` le remplit.
- Un rayon du quart du nombre de places va de 0 à environ 18 pixels (la plus grande station a 70 places).
- Le nombre de places est dans `station["fields"]["bike_stands"]` ; comptez les stations sans vélo dans la boucle.

```python
carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=14)
sans_velo = 0
for station in donnees["records"]:
  champs = station["fields"]
  disponible = champs["available_bikes"] > 0
  sans_velo += not disponible
  folium.CircleMarker(
    location=champs["position"],
    radius=...,
    color="blue" if disponible else ...,
    fill=True,
    tooltip=f"{champs['name']} : {champs['available_bikes']} vélos, {...} places",
  ).add_to(carte)

print(sans_velo, "stations sans vélo")
carte
```

#### Solution

In [ ]:
carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=14)
sans_velo = 0
for station in donnees["records"]:
  champs = station["fields"]
  disponible = champs["available_bikes"] > 0
  sans_velo += not disponible
  folium.CircleMarker(
    location=champs["position"],
    radius=champs["bike_stands"] / 4,
    color="blue" if disponible else "red",
    fill=True,
    tooltip=f"{champs['name']} : {champs['available_bikes']} vélos, "
    f"{champs['bike_stands']} places",
  ).add_to(carte)

# 14 stations sans vélo ; la plus grande, 070-GARE DE NANTES SUD, a 70 places ;
# la borne de test 01011 n'a aucune place : son cercle a un rayon nul.
print(sans_velo, "stations sans vélo")
carte

## Ajout d'une couche GeoJSON

Nous allons maintenant créer une carte plus complexe, avec une couche GeoJSON.

Nous allons pour cela récupérer deux fichiers GeoJSON : `bureaux.geojson` contient les lieux de vote (une école, une salle municipale…) des 21 communes de la métropole, 175 points ; `zones.geojson` contient les zones des 207 bureaux de vote de Nantes, chacune rattachée au lieu de vote où votent ses électeurs, qui peut accueillir plusieurs bureaux.

In [ ]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/nantes-open-data/zones.geojson
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/nantes-open-data/bureaux.geojson

### Chargement des données

Commencez par charger les deux fichiers récupérés à l'aide de la fonction `geopandas.read_file`.

Joignez ensuite à la table des zones celle des lieux de vote, sans sa colonne de géométrie, pour connaître l'adresse du lieu de vote de chaque zone : la colonne `lien_lieu_vote` de la table des zones donne l'identifiant du lieu, la colonne `idlieu_vote` de la table des lieux de vote. Les deux exercices suivants se servent de cette jointure.

**Documentation utile**

- https://geopandas.org/en/stable/docs/reference/api/geopandas.read_file.html
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `geopandas.read_file("zones.geojson")` renvoie une `GeoDataFrame`, une `DataFrame` dont la colonne `geometry` contient les géométries : 207 polygones pour les zones, 175 points pour les lieux de vote.
- `zones_gdf.merge(autre, how="left", left_on="lien_lieu_vote", right_on="idlieu_vote")` associe à chaque zone la ligne de l'autre table dont l'`idlieu_vote` vaut son `lien_lieu_vote`, et garde toutes les zones (`how="left"`) ; le résultat reste une `GeoDataFrame`.
- Retirez la géométrie des lieux de vote avec `drop(columns="geometry")` : sinon le résultat a deux colonnes de géométrie, et `folium.GeoJson` lève `TypeError: Object of type Point is not JSON serializable` à l'exercice suivant.
- Les deux tables ont aussi les colonnes `gid`, `commune`, `code_insee`, `nom` et `site` : `suffixes=("", "_lieu")` garde le nom de celles des zones et suffixe celles des lieux de vote.
- Vous devez obtenir 207 zones, chacune avec une `adresse`.

```python
zones_gdf = geopandas.read_file("zones.geojson")
lieux_gdf = geopandas.read_file("bureaux.geojson")
zones_lieux = zones_gdf.merge(
  lieux_gdf.drop(columns=...),
  how="left",
  left_on=...,
  right_on=...,
  suffixes=("", "_lieu"),
)
print(len(zones_lieux), zones_lieux["adresse"].isna().sum())
```

#### Solution

In [ ]:
zones_gdf = geopandas.read_file("zones.geojson")
lieux_gdf = geopandas.read_file("bureaux.geojson")
zones_lieux = zones_gdf.merge(
  lieux_gdf.drop(columns="geometry"),
  how="left",
  left_on="lien_lieu_vote",
  right_on="idlieu_vote",
  suffixes=("", "_lieu"),
)
print(len(zones_lieux), zones_lieux["adresse"].isna().sum())

### Affichage des zones de vote

Utilisez `folium.GeoJson` pour afficher les zones de vote.

Si vous souhaitez corser l'exercice, affichez au survol de la souris le nom du lieu de vote, le numéro du bureau et l'adresse du lieu de vote (que donne la jointure).

**Documentation utile**

- https://python-visualization.github.io/folium/latest/reference.html#folium.features.GeoJson
- https://python-visualization.github.io/folium/latest/reference.html#folium.features.GeoJsonTooltip

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `folium.GeoJson` prend directement une `GeoDataFrame`, et `name="Zones de vote"` nomme la couche (l'exercice suivant s'en sert).
- `tooltip=folium.GeoJsonTooltip(fields=[...], aliases=[...])` : `fields` liste les colonnes à afficher, `aliases` leurs libellés, dans le même ordre.
- Les colonnes : `nom` (le nom du lieu de vote, que partagent jusqu'à six zones), `numero_bureau` et `adresse`.

```python
carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=13)

folium.GeoJson(
  zones_lieux,
  tooltip=folium.GeoJsonTooltip(fields=[...], aliases=[...]),
  name="Zones de vote",
).add_to(carte)

carte
```

#### Solution

In [ ]:
carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=13)

folium.GeoJson(
  zones_lieux,
  tooltip=folium.GeoJsonTooltip(
    fields=["nom", "numero_bureau", "adresse"],
    aliases=["Lieu de vote", "N° de bureau", "Adresse"],
  ),
  name="Zones de vote",
).add_to(carte)

carte

### Ajout de marqueurs pour les lieux de vote

Repartez du code de la carte précédente et ajoutez un marqueur pour chaque lieu de vote de Nantes, c'est-à-dire chaque lieu de vote qu'une zone désigne. Pour corser l'exercice, ajoutez l'affichage du nom du lieu de vote et de son adresse quand on clique sur le marqueur.

Placez enfin les marqueurs dans une couche à eux, et ajoutez un sélecteur de couches qui affiche ou masque séparément les zones et les marqueurs.

**Documentation utile**

- https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html
- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iterrows.html
- https://python-visualization.github.io/folium/latest/reference.html#folium.map.FeatureGroup
- https://python-visualization.github.io/folium/latest/reference.html#folium.map.LayerControl

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- Gardez les lieux de vote dont l'`idlieu_vote` figure dans la colonne `lien_lieu_vote` de la jointure, avec `isin` : 49 lieux (les 126 autres, hors de Nantes, n'ont pas de zone).
- `iterrows()` donne des couples (index, ligne) ; la géométrie d'un lieu de vote est un point : `lieu.geometry.y` est sa latitude, `lieu.geometry.x` sa longitude, et folium attend (latitude, longitude).
- `popup=` accepte du HTML : `<br />` passe à la ligne.
- `folium.FeatureGroup(name="Lieux de vote")` crée une couche : ajoutez-y les marqueurs (`.add_to(couche_lieux)`) et ajoutez-la à la carte. `folium.LayerControl()` liste les couches nommées de la carte : des marqueurs ajoutés à la carte elle-même n'y figurent pas.

```python
lieux_nantes = lieux_gdf[lieux_gdf["idlieu_vote"].isin(...)]
couche_lieux = folium.FeatureGroup(name="Lieux de vote").add_to(carte)
for _, lieu in lieux_nantes.iterrows():
  folium.Marker(
    location=(..., ...),
    popup=f"{lieu.nom}<br />{...}",
  ).add_to(...)

folium.LayerControl().add_to(carte)
```

#### Solution

In [ ]:
carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=13)

folium.GeoJson(
  zones_lieux,
  tooltip=folium.GeoJsonTooltip(
    fields=["nom", "numero_bureau", "adresse"],
    aliases=["Lieu de vote", "N° de bureau", "Adresse"],
  ),
  name="Zones de vote",
).add_to(carte)

lieux_nantes = lieux_gdf[lieux_gdf["idlieu_vote"].isin(zones_lieux["lien_lieu_vote"])]
couche_lieux = folium.FeatureGroup(name="Lieux de vote").add_to(carte)
for _, lieu in lieux_nantes.iterrows():
  folium.Marker(
    location=(lieu.geometry.y, lieu.geometry.x),
    popup=f"{lieu.nom}<br />{lieu.adresse}",
    icon=folium.Icon(icon="info-sign", color="green"),
  ).add_to(couche_lieux)

folium.LayerControl().add_to(carte)

carte

## Carte choroplèthe des stations (facultatif)

Colorez les zones de vote de Nantes selon le nombre de stations de vélos qu'elles contiennent.

Construisez pour cela une `GeoDataFrame` des stations, des points faits de leurs positions, puis trouvez la zone de chaque station par une jointure spatiale avec `zones_gdf`, comptez les stations de chaque bureau (`idbureau`), et dessinez la carte avec `folium.Choropleth`.

Combien de stations sont dans une zone, et combien de zones ont au moins une station ? Pourquoi certaines stations ne tombent-elles dans aucune zone ?

**Documentation utile**

- https://geopandas.org/en/stable/docs/reference/api/geopandas.points_from_xy.html
- https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `geopandas.points_from_xy(longitudes, latitudes)` construit les points : la longitude d'abord (x), l'ordre de GeoJSON et de shapely, l'inverse de `fields["position"]`. `crs="EPSG:4326"` déclare des degrés WGS 84, ceux de `zones_gdf`.
- `geopandas.sjoin(stations_gdf, zones_gdf[["idbureau", "geometry"]], predicate="within")` garde les stations situées dans une zone, avec l'`idbureau` de leur zone.
- `groupby("idbureau").size()` compte les stations de chaque zone, et `.reset_index(name="stations")` en fait une `DataFrame` à deux colonnes.
- `folium.Choropleth` : `geo_data=zones_gdf`, `data=` les comptes, `columns=["idbureau", "stations"]` (la clef, puis la valeur), `key_on="feature.properties.idbureau"` ; `nan_fill_color="lightgray"` grise les zones sans station.
- Vous devez trouver 117 stations dans une zone, et 65 zones avec au moins une station.

```python
stations = donnees["records"]
stations_gdf = geopandas.GeoDataFrame(
  {"nom": [s["fields"]["name"] for s in stations]},
  geometry=geopandas.points_from_xy(
    [s["fields"]["position"][...] for s in stations],
    [s["fields"]["position"][...] for s in stations],
  ),
  crs="EPSG:4326",
)
dans_une_zone = geopandas.sjoin(stations_gdf, ..., predicate="within")
comptes = dans_une_zone.groupby(...).size().reset_index(name="stations")
print(len(dans_une_zone), "stations dans une zone,", len(comptes), "zones")

carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=13)
folium.Choropleth(
  geo_data=zones_gdf,
  data=comptes,
  columns=[..., ...],
  key_on=...,
  fill_color="BuGn",
  nan_fill_color="lightgray",
  legend_name="Nombre de stations de vélos",
).add_to(carte)
carte
```

### Solution

In [ ]:
stations = donnees["records"]
stations_gdf = geopandas.GeoDataFrame(
  {"nom": [s["fields"]["name"] for s in stations]},
  geometry=geopandas.points_from_xy(
    [s["fields"]["position"][1] for s in stations],
    [s["fields"]["position"][0] for s in stations],
  ),
  crs="EPSG:4326",
)
dans_une_zone = geopandas.sjoin(
  stations_gdf, zones_gdf[["idbureau", "geometry"]], predicate="within"
)
comptes = dans_une_zone.groupby("idbureau").size().reset_index(name="stations")
print(len(dans_une_zone), "stations dans une zone,", len(comptes), "zones")
# Les 12 autres sont hors de Nantes, à Rezé et à Saint-Sébastien-sur-Loire, ou des
# bornes de test.
print(sorted(set(stations_gdf["nom"]) - set(dans_une_zone["nom"])))

carte = folium.Map(location=COORDONNEES_NANTES, zoom_start=13)
folium.Choropleth(
  geo_data=zones_gdf,
  data=comptes,
  columns=["idbureau", "stations"],
  key_on="feature.properties.idbureau",
  fill_color="BuGn",
  nan_fill_color="lightgray",
  legend_name="Nombre de stations de vélos",
).add_to(carte)
carte